# TensorRT

## TensorRT 변환 흐름과 최적화

1. Torch to TensorRT
    1. ONNX grpah 변환
    2. ONNX 모델을 tensorRT 그래프로 변환 

- nvinfer1::ICudaEngine *engine; : 학습한 Network를 Inference할 TRT engine 입니다.
    - '네트워크 정의'와 '학습된 파라미터'들이 정의된 엔진입니다.
    - 엔진은 작업이 끝날 때까지 반납되어서는 안됩니다.
- nvinfer1::IExecutionContext* context[L]; : CUDA context는 CPU thread와 같이 작업 단위입니다.
    - 미리 정의된 환경에서 실행하기 위한 환경입니다. Inference중간에 생기는 activation 값들 또한 이 Context에서 관리된다고 생각하시면 됩니다.
    - 1개의 engine을 위한 여러개의 context가 존재할 수 있으며, 이는 동일한 네트워크를 동시에 여러개의 Input에 대해서 실행하는 경우(여러개 Batch와 같은) 활용이 가능합니다. 이때, weight는 공유되어 쓰일 수 있다고 합니다.**
    - 1개의 Context에 대한 2개 이상의 kernel processing은 concurrently 실행이 불가하지만, 여러개 context에 대한 kernel들은 concurrently 실행이 가능합니다.
- cudaStream_t stream[L]; : 일반적인 cuda stream으로 inference kernel이 쌓일 Queue입니다.
    - Resource가 충분해지면, 메모리 복사와 Kernel 실행이 모두 진행됩니디다.
    - 두개 이상의 stream은 가능하지만, 두개 이상의 stream을 가지면 unordered형태로 overlap 가능합니다.
    - ‘Consecutive’ computation in device
    - Can work with other streams concurrently
- float *CPUinput, CPUoutput : input과 output을 담을 CPU memory
- void *GPUinput, GPUoutput : input과 output을 담을 GPU memory
    - GPU는 input과 output은 하나의 버퍼로 선언한 뒤 "engine-> getBindingIndex("input:0")" 함수를 통해 해당 buffer의 인덱스를 얻어내 사용하는 경우도 있습니다.
- nvinfer1::iLogger gLogger : logging을 위한 모든 class instance들을 담기 위한 오브젝트
- nvinfer1::iRuntime : serialized 된 nvinfer1::ICudaEngine을 deserialize 한 오브젝트입니다.
    - setDLAcore() : 사용할 DLA 코어의 개수를 명시할 수 있습니다.
    - * 왜 serialize하고 나서 deserialize하는가? Inference를 위해서 꼭 serialize & deserialize를 해야만 하는 것은 아닙니다. 하지만 빌드되는 시간을 줄이기 위해 바로 사용하지 않는 경우 deserialize해 저장해두는 것이 유리하기 때문에 주로 serialize 해 저장해두곤 합니다.

![[Engine-Context-Stream 관계]](https://blog.kakaocdn.net/dn/0xMdK/btrUb476xpx/9Xi9N5ckZ9XIqY8t6It9c1/img.png)

[Engine-Context-Stream 관계]
